In [4]:
import pandas as pd

import hashlib as hashlib

import numpy as np


In [5]:
file_path = "../Files/raw/healthcare_dataset.csv"
data = pd.read_csv(file_path)

In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55500 entries, 0 to 55499
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Name                55500 non-null  object 
 1   Age                 55500 non-null  int64  
 2   Gender              55500 non-null  object 
 3   Blood Type          55500 non-null  object 
 4   Medical Condition   55500 non-null  object 
 5   Date of Admission   55500 non-null  object 
 6   Doctor              55500 non-null  object 
 7   Hospital            55500 non-null  object 
 8   Insurance Provider  55500 non-null  object 
 9   Billing Amount      55500 non-null  float64
 10  Room Number         55500 non-null  int64  
 11  Admission Type      55500 non-null  object 
 12  Discharge Date      55500 non-null  object 
 13  Medication          55500 non-null  object 
 14  Test Results        55500 non-null  object 
dtypes: float64(1), int64(2), object(12)
memory usage: 6.4

In [7]:
data.head(5)

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal


Standardizing names and doctor names to handle casing inconsistencies (e.g., "Bobby JackSon", "LesLie TeRRy")

In [8]:
data['Name_Clean'] = data['Name'].astype(str).str.strip().str.title()
data['Doctor_Clean'] = data['Doctor'].astype(str).str.strip().str.title()

Mapping Unique Patient IDs (Consistent for repeating patients)

In [9]:
unique_patients = {name: f"PAT_{idx+10001:05d}" for idx, name in enumerate(data['Name_Clean'].unique())}
data['Patient_ID'] = data['Name_Clean'].map(unique_patients)

Mapping Unique Provider IDs (Consistent per Doctor)

In [10]:
unique_doctors = {doc: f"PRV_{idx+5001:04d}" for idx, doc in enumerate(data['Doctor_Clean'].unique())}
data['Provider_ID'] = data['Doctor_Clean'].map(unique_doctors)

Generating Unique Claim Numbers for each record

In [11]:
data['Claim_Number'] = [f"CLM_{idx+100001:06d}" for idx in range(len(data))]

Dropping direct identifiers (Patient Name & Doctor Name) and temporary clean columns

In [12]:
data = data.drop(columns=['Name', 'Doctor', 'Name_Clean', 'Doctor_Clean'])

Reordering columns logically (putting identifiers at the front)

In [13]:
column_order = ['Claim_Number', 'Patient_ID', 'Provider_ID'] + [col for col in data.columns if col not in ['Claim_Number', 'Patient_ID', 'Provider_ID']]
data = data[column_order]

In [14]:
data.head(5)

,Claim_Number,Patient_ID,Provider_ID,Age,Gender,Blood Type,Medical Condition,Date of Admission,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,CLM_100001,PAT_10001,PRV_5001,30,Male,B-,Cancer,2024-01-31,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,CLM_100002,PAT_10002,PRV_5002,62,Male,A+,Obesity,2019-08-20,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,CLM_100003,PAT_10003,PRV_5003,76,Female,A-,Obesity,2022-09-22,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,CLM_100004,PAT_10004,PRV_5004,28,Female,O+,Diabetes,2020-11-18,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,CLM_100005,PAT_10005,PRV_5005,43,Female,AB+,Cancer,2022-09-19,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal
